# Module 19 — Capstone Project: Personal Finance Tracker

**Part IV: Advanced** · Level: Advanced · Prerequisites: Modules 1–18

## The goal

Build a small, well-structured application that tracks income and expenses, saves them to a CSV file, summarises spending by category, and ships with automated tests and documentation. The project deliberately exercises every part of the course:

| Skill | Where it appears |
|---|---|
| Data types, f-strings, formatting | reports and text chart |
| Collections and comprehensions | grouping and sorting transactions |
| Functions and type hints | every public function |
| Exceptions and custom errors | validation of input and bad rows |
| Files, CSV, `pathlib` | persistence |
| Dataclasses and OOP | `Transaction`, `Tracker` |
| Generators | lazy month filtering |
| Modules and `__main__` guard | a script that is also a library |
| Testing | `unittest` suite |
| Documentation | README |

Everything uses the **standard library only**, so it runs anywhere Python 3.10+ is installed.

## Requirements

1. A transaction has a date, a kind (`income` or `expense`), a category, a description, and a positive amount.
2. Invalid transactions are rejected with a clear `ValidationError`.
3. Transactions persist in a CSV file and reload on start-up.
4. The tracker reports total income, total expenses, and balance.
5. It summarises expenses by category, largest first, and can draw a simple text bar chart.
6. It can list the transactions of any given month without building a second full list.
7. The code is tested, documented, and importable without side effects.

## Step 1 — Design before coding

Decide the pieces first:

- `FinanceError` → `ValidationError` — a small exception family
- `Transaction` — an immutable, validated record (a frozen dataclass)
- `Tracker` — owns the list of transactions and the file
- Pure helper behaviour lives in methods that *return* data; only `main()` prints

## Step 2 — The implementation

Save this as `finance_tracker.py`.

In [ ]:
"""finance_tracker.py -- a small personal finance tracker (standard library only)."""
from __future__ import annotations

import csv
from collections import defaultdict
from dataclasses import asdict, dataclass
from datetime import date
from pathlib import Path
from typing import Iterator

FIELDS = ["date", "kind", "category", "description", "amount"]
KINDS = {"income", "expense"}


class FinanceError(Exception):
    """Base class for all errors raised by this module."""


class ValidationError(FinanceError):
    """Raised when a transaction or a stored row is invalid."""


@dataclass(frozen=True)
class Transaction:
    """One validated income or expense record."""

    date: date
    kind: str
    category: str
    description: str
    amount: float

    def __post_init__(self) -> None:
        if self.kind not in KINDS:
            raise ValidationError(f"kind must be one of {sorted(KINDS)}, got {self.kind!r}")
        if not self.category.strip():
            raise ValidationError("category cannot be empty")
        if self.amount <= 0:
            raise ValidationError(f"amount must be positive, got {self.amount}")

    @classmethod
    def from_row(cls, row: dict[str, str]) -> "Transaction":
        """Build a Transaction from a dict of strings (e.g. a CSV row)."""
        try:
            return cls(
                date=date.fromisoformat(row["date"]),
                kind=row["kind"].strip().lower(),
                category=row["category"].strip().lower(),
                description=row.get("description", "").strip(),
                amount=float(row["amount"]),
            )
        except (KeyError, ValueError) as err:
            raise ValidationError(f"invalid transaction {row!r}: {err}") from err

    def to_row(self) -> dict[str, str]:
        row = asdict(self)
        row["date"] = self.date.isoformat()
        row["amount"] = f"{self.amount:.2f}"
        return row


class Tracker:
    """A collection of transactions backed by a CSV file."""

    def __init__(self, path: str | Path) -> None:
        self.path = Path(path)
        self.transactions: list[Transaction] = []
        if self.path.exists():
            self.load()

    # -- persistence -------------------------------------------------------
    def load(self) -> None:
        with open(self.path, newline="", encoding="utf-8") as f:
            self.transactions = [Transaction.from_row(row) for row in csv.DictReader(f)]

    def save(self) -> None:
        with open(self.path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=FIELDS)
            writer.writeheader()
            writer.writerows(t.to_row() for t in self.transactions)

    # -- commands ----------------------------------------------------------
    def add(self, when: str, kind: str, category: str, description: str, amount: float) -> Transaction:
        """Validate, store, and save a new transaction."""
        transaction = Transaction.from_row(
            {"date": when, "kind": kind, "category": category,
             "description": description, "amount": str(amount)}
        )
        self.transactions.append(transaction)
        self.save()
        return transaction

    # -- queries -----------------------------------------------------------
    def totals(self) -> tuple[float, float, float]:
        """Return (income, expenses, balance)."""
        income = sum(t.amount for t in self.transactions if t.kind == "income")
        expenses = sum(t.amount for t in self.transactions if t.kind == "expense")
        return income, expenses, income - expenses

    def spending_by_category(self) -> dict[str, float]:
        """Expenses grouped by category, largest first."""
        grouped: defaultdict[str, float] = defaultdict(float)
        for t in self.transactions:
            if t.kind == "expense":
                grouped[t.category] += t.amount
        return dict(sorted(grouped.items(), key=lambda item: item[1], reverse=True))

    def in_month(self, year: int, month: int) -> Iterator[Transaction]:
        """Lazily yield the transactions that fall in the given month."""
        return (t for t in self.transactions if (t.date.year, t.date.month) == (year, month))

    def text_chart(self, width: int = 30) -> str:
        """A bar chart of spending by category, drawn with '#' characters."""
        data = self.spending_by_category()
        if not data:
            return "(no expenses recorded)"
        largest = max(data.values())
        return "\n".join(
            f"{category:<12}{'#' * round(width * amount / largest):<{width}} {amount:>9,.2f}"
            for category, amount in data.items()
        )

    def report(self) -> str:
        income, expenses, balance = self.totals()
        return (
            f"Income:   {income:>10,.2f}\n"
            f"Expenses: {expenses:>10,.2f}\n"
            f"Balance:  {balance:>10,.2f}"
        )


def main() -> None:
    """Small demonstration using a temporary file."""
    import tempfile

    with tempfile.TemporaryDirectory() as folder:
        tracker = Tracker(Path(folder) / "finance.csv")
        tracker.add("2026-10-01", "income", "salary", "October pay", 3200)
        tracker.add("2026-10-02", "expense", "rent", "Monthly rent", 1100)
        tracker.add("2026-10-03", "expense", "food", "Groceries", 84.50)
        tracker.add("2026-10-07", "expense", "food", "Restaurant", 42.00)
        tracker.add("2026-10-09", "expense", "transport", "Metro pass", 60)

        print(tracker.report())
        print()
        print(tracker.text_chart())
        print()
        print("October transactions:", sum(1 for _ in tracker.in_month(2026, 10)))

        try:
            tracker.add("2026-10-10", "expense", "food", "Refund?", -5)
        except ValidationError as err:
            print("Rejected:", err)


if __name__ == "__main__":
    main()

Run it with `python finance_tracker.py`. Expected output (abridged):

```text
Income:     3,200.00
Expenses:   1,286.50
Balance:    1,913.50

rent        ##############################  1,100.00
food        ###                               126.50
transport   ##                                 60.00
...
```

## Step 3 — Tests

Save as `test_finance_tracker.py` in the same folder.

In [ ]:
import tempfile
import unittest
from datetime import date
from pathlib import Path

from finance_tracker import Tracker, Transaction, ValidationError


class TransactionTests(unittest.TestCase):
    def test_valid_transaction(self):
        t = Transaction(date(2026, 10, 1), "income", "salary", "pay", 100.0)
        self.assertEqual(t.amount, 100.0)

    def test_rejects_non_positive_amount(self):
        with self.assertRaises(ValidationError):
            Transaction(date(2026, 10, 1), "expense", "food", "x", 0)

    def test_rejects_unknown_kind(self):
        with self.assertRaises(ValidationError):
            Transaction(date(2026, 10, 1), "gift", "food", "x", 5)

    def test_from_row_rejects_bad_date(self):
        row = {"date": "not-a-date", "kind": "income", "category": "a", "amount": "1"}
        with self.assertRaises(ValidationError):
            Transaction.from_row(row)


class TrackerTests(unittest.TestCase):
    def setUp(self):
        self._dir = tempfile.TemporaryDirectory()
        self.path = Path(self._dir.name) / "data.csv"
        self.tracker = Tracker(self.path)
        self.tracker.add("2026-10-01", "income", "salary", "pay", 1000)
        self.tracker.add("2026-10-02", "expense", "food", "lunch", 200)
        self.tracker.add("2026-11-02", "expense", "rent", "rent", 300)

    def tearDown(self):
        self._dir.cleanup()

    def test_totals(self):
        self.assertEqual(self.tracker.totals(), (1000.0, 500.0, 500.0))

    def test_spending_by_category_is_sorted(self):
        self.assertEqual(list(self.tracker.spending_by_category()), ["rent", "food"])

    def test_persistence_round_trip(self):
        reloaded = Tracker(self.path)
        self.assertEqual(reloaded.transactions, self.tracker.transactions)

    def test_in_month_filters_lazily(self):
        october = list(self.tracker.in_month(2026, 10))
        self.assertEqual(len(october), 2)

    def test_chart_handles_empty(self):
        empty = Tracker(Path(self._dir.name) / "empty.csv")
        self.assertIn("no expenses", empty.text_chart())


if __name__ == "__main__":
    unittest.main()

Run them with:

In [ ]:
%%bash
python -m unittest -v

## Step 4 — Documentation

Write a `README.md` so someone else can use the project:

In [ ]:
%%markdown
# Personal Finance Tracker

Track income and expenses, summarise spending by category, and keep your data in a plain CSV file.

## Requirements
Python 3.10 or newer. No third-party packages.

## Usage
```python
from finance_tracker import Tracker

tracker = Tracker("my_finances.csv")
tracker.add("2026-10-01", "income", "salary", "October pay", 3200)
tracker.add("2026-10-02", "expense", "rent", "Monthly rent", 1100)
print(tracker.report())
print(tracker.text_chart())
```

## Running the tests
`python -m unittest -v`

## Project layout
- `finance_tracker.py` — the library and demo
- `test_finance_tracker.py` — automated tests

## Step 5 — Extend it (choose two)

1. **Command-line interface** with `argparse`: `python finance_tracker.py add --kind expense ...`.
2. **Budgets:** a `set_budget(category, limit)` method and a warning when spending exceeds it, using `logging`.
3. **Monthly summary:** a generator that yields `(year, month, income, expenses)` for every month in the data.
4. **JSON export/import** next to CSV, with a shared interface (`Storage` base class, two subclasses).
5. **Recurring transactions** (rent, subscriptions) generated with `datetime` and `timedelta`.
6. **Charts:** if you have `matplotlib` installed, add a `plot_spending()` method — keep it an *optional* import so the core stays dependency-free.
7. **Type-check it:** run `mypy` and fix every report.

## Assessment checklist

- [ ] All tests pass, and you added at least three of your own
- [ ] Invalid input raises `ValidationError`, never a bare traceback
- [ ] No function both calculates and prints
- [ ] Every public function and class has a docstring
- [ ] The module can be imported without printing anything
- [ ] Code is formatted with `black` and clean under `ruff`
- [ ] You can explain each design decision (why a frozen dataclass? why a generator for `in_month`?)

## Where to go next

- **Web and APIs:** `requests`, FastAPI or Flask
- **Data analysis:** `pandas`, `numpy`, `matplotlib`
- **Automation and scripting:** `argparse`, `subprocess`, `schedule`
- **Concurrency:** `threading`, `multiprocessing`, `asyncio`
- **Packaging:** `pyproject.toml`, building and publishing a package
- **Specialised domains:** machine learning, scientific computing, or any library you want to master — you now have the foundations to read its documentation and its source code.